# Configuration Bundle A/B Testing with Amazon Bedrock AgentCore

This notebook demonstrates **Pattern 2: Configuration Bundle Variants** — testing different system prompts on the **same runtime** without deploying separate agents.

📚 [Configuration Bundle A/B Testing Guide](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/ab-testing-config-bundle.html) | [A/B Testing Overview](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/ab-testing.html)

### How it differs from target-based (Lab 1)

| | Target-based (Lab 1) | Config bundle (this lab) |
|---|---|---|
| Agents deployed | 2 separate runtimes | 1 runtime with config hook |
| What varies | Model + code + prompt | System prompt only |
| Mechanism | Different gateway targets | Config bundles via W3C baggage headers |
| Eval config | Per-variant (2 configs) | Shared (1 config) |
| Infrastructure cost | 2 runtimes | 1 runtime |
| Iteration speed | Slow (redeploy) | Fast (update bundle) |
| Use case | Framework/model changes | Prompt engineering |

### Use Case: FixFirst Appliance Support Agent — Prompt Engineering

Same model (Amazon Nova Lite), same code, but two different system prompts:

| | Control (C) | Treatment (T1) |
|---|---|---|
| **Model** | Amazon Nova Lite | Amazon Nova Lite |
| **Prompt style** | Conversational, one question at a time | Structured IDENTIFY/DIAGNOSE/RESOLVE |
| **Cost** | Same | Same |

### How it works

The agent uses a `BeforeModelCallEvent` hook that reads the active configuration bundle from `BedrockAgentCoreContext.get_config_bundle()`. During an A/B test, the Gateway assigns each session to a variant and propagates the corresponding bundle reference via W3C baggage headers. The runtime applies whichever system prompt is in the assigned bundle.

### What We Expect to Find

- The structured prompt (IDENTIFY/DIAGNOSE/RESOLVE) should produce more helpful, actionable responses
- If statistically significant (p < 0.05), we can improve the agent by simply updating the system prompt — no model upgrade or code change needed
- This is the cheapest possible improvement: same model, same infrastructure, just a better prompt

### Architecture

<pre>
+-------------------------------------------------------+
|            AgentCore Gateway (IAM Auth)               |
|                                                       |
|  +--------------------------------------------------+ |
|  |     A/B Test (50/50 config bundle split)         | |
|  |  Session -> Control bundle OR Treatment bundle   | |
|  +------------------------+-------------------------+ |
+---------------------------|---------------------------+
                            |
                  +---------+---------+
                  | Target: fixfirst  |  (single target)
                  +---------+---------+
                            |
                  +---------+---------------------+
                  | AgentCore Runtime (Nova Lite) |
                  |                               |
                  | BeforeModelCallEvent hook:    |
                  | reads config bundle from      |
                  | baggage headers, applies      |
                  | system prompt dynamically     |
                  +---------+---------------------+
                            |
                       OTel spans
                            |
                  +---------+---------------------+
                  | Shared Online Eval Config     |
                  | Builtin.Helpfulness           |
                  | (scores both variants from    |
                  |  the same log group)          |
                  +---------+---------------------+
                            |
                  +---------+---------------------+
                  | A/B Test Aggregation          |
                  | mean, p-value, CI, sig        |
                  +-------------------------------+
</pre>

**Step by step:**

1. The **client** sends a SigV4-signed HTTP request to the gateway. Each request includes a unique session ID.
2. The **A/B test** assigns the session to either the control or treatment **configuration bundle**. Unlike target-based routing, both variants go to the same target — only the config bundle reference differs.
3. The **gateway** forwards the request to the single target (`fixfirst`), injecting W3C `baggage` headers containing the assigned bundle ARN and version.
4. The **AgentCore Runtime** receives the request. The `BeforeModelCallEvent` hook calls `BedrockAgentCoreContext.get_config_bundle()` which reads the bundle reference from the baggage headers, fetches the bundle content, and applies the `system_prompt` value before each LLM call.
5. After 15 minutes of inactivity, the session completes. The **shared online evaluator** scores it — both variants write to the same log group, but the evaluator knows which variant each session belongs to (from the baggage metadata on the spans).
6. The **A/B test aggregation pipeline** separates scores by variant and computes statistics: which prompt produces more helpful responses, with what confidence.

**Estimated time:** ~30 minutes (including ~15 min wait for evaluation results)

**Prerequisites:** Python 3.12+, `uv`, Node.js, AWS CLI >= 2.34, CDK bootstrapped, Bedrock model access for `amazon.nova-lite-v1:0`, `bash_kernel`.

**Note:** This notebook uses the **Bash kernel**. Select *Kernel → Change Kernel → Bash* if not already selected.

## 1. Check Prerequisites

In [ ]:
cd ~/sample-aws-agentops-fix-first-agent/ab_testing
~/sample-aws-agentops-fix-first-agent/ab_testing/scripts/check_prerequisites.sh

## 2. Package and Deploy the Agent

Unlike target-based testing (which requires 2 runtimes), configuration bundle testing uses a **single agent** that dynamically reads its system prompt from the assigned config bundle.

The key code in `main.py`:
```python
def dynamic_config_hook(event: BeforeModelCallEvent):
    config = BedrockAgentCoreContext.get_config_bundle()
    if config:
        event.agent.system_prompt = config.get("system_prompt", DEFAULT_SYSTEM_PROMPT)

agent.hooks.add_callback(BeforeModelCallEvent, dynamic_config_hook)
```

📚 [Configuration Bundles at Runtime](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/configuration-bundles-runtime.html)

### 2.1 Package Agent

In [ ]:
~/sample-aws-agentops-fix-first-agent/ab_testing/configuration_based_variants/scripts/package_config_agent.sh

### 2.2 Deploy Runtime + Evaluation Config

Deploys `fixFirstAgent-ConfigABTestingStack` via CDK:
- One AgentCore Runtime with the config-bundle-aware agent
- One shared Online Evaluation Config (`Builtin.Helpfulness`)
- IAM roles (including `bedrock-agentcore:GetConfigurationBundleVersion` permission for reading bundles)
- SSM parameters for resource ARNs

In [ ]:
~/sample-aws-agentops-fix-first-agent/ab_testing/configuration_based_variants/scripts/deploy_config_agent.sh

### 2.3 Wait for Runtime to Become READY

In [ ]:
REGION=${AWS_REGION:-$(aws configure get region 2>/dev/null)}
REGION=${REGION:-us-east-1}
RUNTIME_ARN=$(aws ssm get-parameter --name /fixFirstAgent/config-runtime-arn --query Parameter.Value --output text --region $REGION)
RUNTIME_ID=$(echo "$RUNTIME_ARN" | awk -F/ '{print $NF}')

echo "Waiting for runtime ($RUNTIME_ID)..."
for i in $(seq 1 30); do
    STATUS=$(aws bedrock-agentcore-control get-agent-runtime --agent-runtime-id "$RUNTIME_ID" --region $REGION --query status --output text 2>/dev/null)
    if [ "$STATUS" = "READY" ]; then
        echo "  Runtime is READY"
        break
    fi
    echo "  [$i/30] $STATUS"
    sleep 20
done
if [ "$STATUS" != "READY" ]; then
    echo "ERROR: Runtime not READY after 10 minutes"
    exit 1
fi

## 3. Create Config Bundles, Gateway, and A/B Test

This step creates all the A/B test infrastructure via a Python script (these APIs aren't yet in CloudFormation):

1. **Two configuration bundles** — each contains a different `system_prompt` value
2. **One AgentCore Gateway** with a single HTTP target pointing to the runtime
3. **An A/B test** with `variantConfiguration.configurationBundle` (50/50 split)

The script accepts `--control-prompt` and `--treatment-prompt` arguments, so you can easily test different prompts without editing code.

📚 [Gateway Concepts](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway-core-concepts.html)

In [ ]:
REGION=${AWS_REGION:-$(aws configure get region 2>/dev/null)}
export AWS_REGION=${REGION:-us-east-1}
export APP_NAME=fixFirstAgent

python3 ~/sample-aws-agentops-fix-first-agent/ab_testing/configuration_based_variants/scripts/create_config_ab_test.py

## 4. Send Traffic Through Gateway

Sends 20 appliance troubleshooting prompts. The A/B test assigns each session to either the control or treatment config bundle. The runtime reads the assigned bundle via baggage headers and applies the corresponding system prompt before each LLM call.

Note: the target path is `/fixfirst/invocations` (single target named `fixfirst`).

In [ ]:
REGION=${AWS_REGION:-$(aws configure get region 2>/dev/null)}
REGION=${REGION:-us-east-1}
GATEWAY_URL=$(aws ssm get-parameter --name /fixFirstAgent/config-ab-gateway-url --query Parameter.Value --output text --region $REGION)
cd ~/sample-aws-agentops-fix-first-agent/ab_testing
~/sample-aws-agentops-fix-first-agent/ab_testing/scripts/send_traffic.sh "$GATEWAY_URL" "$REGION" ~/sample-aws-agentops-fix-first-agent/ab_testing/prompts.txt "/fixfirst/invocations"

## 5. Check A/B Test Results

Re-run after ~20 minutes. Same interpretation as Lab 1:
- `isSignificant: true` + positive change → structured prompt wins, update the default prompt
- `isSignificant: false` → need more data, send more traffic

📚 [Managing A/B Tests](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/ab-testing-manage.html)

In [ ]:
python3 ~/sample-aws-agentops-fix-first-agent/ab_testing/scripts/check_ab_results.py config-ab-test-id || python ~/sample-aws-agentops-fix-first-agent/ab_testing/scripts/check_ab_results.py config-ab-test-id

## 6. Stop the A/B Test

In [ ]:
REGION=${AWS_REGION:-$(aws configure get region 2>/dev/null)}
REGION=${REGION:-us-east-1}
AB_TEST_ID=$(aws ssm get-parameter --name /fixFirstAgent/config-ab-test-id --query Parameter.Value --output text --region $REGION)
aws bedrock-agentcore update-ab-test --ab-test-id "$AB_TEST_ID" --execution-status STOPPED --region $REGION
echo "A/B test $AB_TEST_ID stopped."

## 7. Cleanup

Removes all resources: A/B test, config bundles, gateway, eval config, IAM roles, CDK stack.

In [ ]:
REGION=${AWS_REGION:-$(aws configure get region 2>/dev/null)}
export APP_NAME=fixFirstAgent
export AWS_REGION=${REGION:-us-east-1}
~/sample-aws-agentops-fix-first-agent/ab_testing/configuration_based_variants/scripts/cleanup_config_all.sh

## Running the Full Workflow Without a Notebook

You can run the entire configuration-based A/B testing workflow end-to-end as a single script:

```bash
./run_config_ab_testing.sh
```

With custom prompts:
```bash
CONTROL_PROMPT="Be brief and friendly" \
TREATMENT_PROMPT="Use step-by-step diagnostic methodology" \
./run_config_ab_testing.sh
```

This performs all steps (prerequisites → package → deploy → wait → create bundles → send traffic → poll for results) and prints the final recommendation.

## Next Steps

### When to use which pattern

| Change you want to test | Pattern | Why |
|---|---|---|
| System prompt wording | Config bundle (this lab) | Prompt is a config value — no redeploy needed |
| Model ID (e.g., Nova Lite → Claude) | Config bundle | Model ID can be a bundle field read by the hook |
| Temperature / top-p / max tokens | Config bundle | Model parameters are configuration |
| Adding a new tool (e.g., order lookup API) | Target-based (Lab 1) | New tool requires code: definition, handler, permissions |
| Switching frameworks (Strands → LangGraph) | Target-based (Lab 1) | Entirely different agent code and dependencies |
| Adding RAG retrieval before responding | Target-based (Lab 1) | New code path: embed query → search index → inject context |

### Example: testing a new tool requires target-based

Suppose you want to test whether giving the agent access to a **parts inventory API** improves helpfulness. The treatment agent needs:

```python
# New tool definition (code change)
@tool
def check_parts_availability(appliance: str, part_name: str) -> str:
    """Check if a replacement part is in stock."""
    response = requests.get(f"{INVENTORY_API}/parts", params={...})
    return response.json()

# Agent wired with the new tool
agent = Agent(model=MODEL, tools=[lookup_manual, check_parts_availability])
```

This can't be expressed as a config bundle — it's new Python code, a new dependency, and new IAM permissions. You'd deploy it as a separate runtime and use **target-based A/B testing** (Lab 1) to measure whether part availability info actually helps customers.

### Combining both patterns

A practical workflow:
1. **Target-based** — pick the best architecture (e.g., with vs without RAG, with vs without a tool)
2. **Config bundle** — optimize the winning architecture's prompt, model, and parameters without redeploying

📚 [A/B Testing Overview](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/ab-testing.html) | [Target-Based Guide](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/ab-testing-target-based.html)